> **Separate residual experiment.** All-noise AHC experiment; not incorporated in the Pass-1-only paper.
> Open [Current_Study_Reproduction.ipynb](../../../../../notebooks/Current_Study_Reproduction.ipynb) for the current manuscript workflow. Original code and outputs below are preserved for provenance.


# Full-noise agglomerative clustering — 35,724 texts
Run cells in order in Google Colab. This memory-safe exact implementation uses a disk-backed float64 condensed array and fastcluster 1.3.0 compiled in-place linkage. It checks current system memory against a conservative 10.5 GiB ceiling. No GPU is required. If installation changes NumPy, restart the runtime and rerun from the top before loading embeddings.

Upload only `noise_embeddings.npz` when prompted. It contains existing text embeddings, no source text, usernames or platform IDs. Embeddings remain research data: keep the notebook/input private. The row-to-record mapping stays on your computer.

This runs **average and complete linkage on every noise text**, cosine distance in original 384D space, cuts 0.40/0.50/0.60. No sampling, UMAP refit, forced cluster count, semantic labels or DLD selection. Groups are review aids, not SMDIs. Download the results ZIP at the end and return it to the project for local joining.

Graphify (635 nodes, 1,186 edges) supplies historical provenance only. See [SciPy linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) and [Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
%pip install -q "fastcluster==1.3.0" "psutil==7.2.2"
import numpy as np, scipy, psutil
import sys, json, hashlib
from pathlib import Path
print({"python":sys.version,"numpy":np.__version__,"scipy":scipy.__version__})


In [ ]:
from google.colab import files
INPUT_SHA256 = '85a4014ac4df55b16ed8fb1cfa3ecd1ee32df0c487961c318970a104d323fdb7'
uploaded = files.upload()
assert 'noise_embeddings.npz' in uploaded, 'Select noise_embeddings.npz from the supplied package.'
assert hashlib.sha256(uploaded['noise_embeddings.npz']).hexdigest() == INPUT_SHA256, 'Wrong or modified input.'
del uploaded
with np.load('noise_embeddings.npz',allow_pickle=False) as data:
    embeddings=np.asarray(data['embeddings'],dtype=np.float64)
assert embeddings.shape == (35724,384) and np.isfinite(embeddings).all()
assert np.all(np.linalg.norm(embeddings,axis=1)>0)
embeddings /= np.linalg.norm(embeddings,axis=1,keepdims=True)
print('Loaded ALL',len(embeddings),'unique noise texts.')


In [ ]:
import csv, gc, hashlib, json, os, shutil, sys, tempfile, threading, time, zipfile
from pathlib import Path
import numpy as np
import scipy
import psutil
import fastcluster
from scipy.spatial.distance import cdist
from scipy.cluster.hierarchy import fcluster, is_valid_linkage

GIB = 2**30

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def _fill_cosine(D, E, block_rows=128):
    """Fill a condensed array; at most block_rows x (n-1) temporary distances."""
    n = len(E)
    next_report = 0.0
    for start in range(0, n - 1, block_rows):
        stop = min(start + block_rows, n - 1)
        block = cdist(E[start:stop], E[start + 1:], metric='cosine')
        np.clip(block, 0, 2, out=block)
        if not np.isfinite(block).all():
            raise ValueError('Non-finite cosine distance; no texts were removed.')
        for i in range(start, stop):
            pos = n * i - i * (i + 1) // 2
            D[pos:pos + n - i - 1] = block[i - start, i - start:]
        del block
        fraction = (n * stop - stop * (stop + 1) // 2) / len(D)
        if fraction >= next_report or stop == n - 1:
            print(f'Distances: {fraction:.1%} of {len(D):,} pairs', flush=True)
            next_report = fraction + 0.10
    D.flush()

def _inplace_linkage(D, n, method):
    # 1.3.0's public Python wrapper first calls np.array(X), making a copy.
    # Its compiled bridge operates directly on the supplied writable buffer.
    # Pin the version: this is a deliberately narrow use of an internal API.
    if fastcluster.__version__ != '1.3.0':
        raise RuntimeError('This adapter requires fastcluster==1.3.0.')
    assert method in ('average', 'complete')
    assert D.dtype == np.float64 and D.ndim == 1
    assert D.flags.c_contiguous and D.flags.aligned and D.flags.writeable
    assert len(D) == n * (n - 1) // 2
    Z = np.empty((n - 1, 4), dtype=np.float64)
    fastcluster.linkage_wrap(n, D, Z, fastcluster.mthidx[method])
    is_valid_linkage(Z, throw=True)
    assert Z[-1, 3] == n
    return Z

def run_full_ahc(embeddings, output_dir, input_sha256):
    E = np.asarray(embeddings, dtype=np.float64, order='C')
    if E.shape != (35724, 384):
        raise ValueError(f'Expected ALL 35,724 x 384 embeddings; got {E.shape}.')
    if not np.isfinite(E).all() or np.any(np.linalg.norm(E, axis=1) == 0):
        raise ValueError('Invalid embeddings; no texts were removed.')
    if fastcluster.__version__ != '1.3.0':
        raise RuntimeError('Install fastcluster==1.3.0 before running.')
    n = len(E)
    pair_bytes = n * (n - 1) // 2 * 8
    block_bytes = 128 * (n - 1) * 8
    # Count the entire mapped file as resident: do not rely on OS eviction.
    additional = pair_bytes + 2 * block_bytes + GIB
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    process = psutil.Process()
    print(f'ALL {n:,} texts; {n*(n-1)//2:,} pairs; float64 cosine', flush=True)
    print(f'One disk-backed array: {pair_bytes/GIB:.2f} GiB; '
          f'conservative additional RAM budget: {additional/GIB:.2f} GiB', flush=True)
    rows, partitions, names = [], {}, []
    versions = {'python': sys.version, 'numpy': np.__version__,
                'scipy': scipy.__version__, 'fastcluster': fastcluster.__version__,
                'psutil': psutil.__version__}
    for method in ['average', 'complete']:
        gc.collect()
        rss = process.memory_info().rss
        memory = psutil.virtual_memory()
        available = memory.available
        predicted_rss = rss + additional
        predicted_system = memory.total - available + additional
        print(f'{method}: available RAM {available/GIB:.2f} GiB; '
              f'conservative system-use estimate {predicted_system/GIB:.2f} GiB', flush=True)
        if max(predicted_rss, predicted_system) > 10.5 * GIB or available < additional + 0.5 * GIB:
            raise RuntimeError('RAM safety check failed. Release other large notebook '
                               'arrays or restart the runtime, reload inputs, and retry. '
                               'Do not reduce the corpus or bypass this check.')
        if shutil.disk_usage(out).free < pair_bytes + GIB:
            raise RuntimeError('Need at least 5.76 GiB of free temporary disk.')
        fd, filename = tempfile.mkstemp(prefix='ahc_dist_', suffix='.f64', dir=out)
        os.close(fd)
        D = None
        started = time.time()
        stop_monitor = threading.Event()
        peak = [rss]
        stage = ['distances']
        def monitor():
            last = time.monotonic()
            while not stop_monitor.wait(0.5):
                peak[0] = max(peak[0], process.memory_info().rss)
                if time.monotonic() - last >= 30:
                    print(f'{method}: {stage[0]}, elapsed {time.time()-started:.0f}s, '
                          f'RSS {process.memory_info().rss/GIB:.2f} GiB', flush=True)
                    last = time.monotonic()
        thread = threading.Thread(target=monitor, daemon=True)
        thread.start()
        try:
            D = np.memmap(filename, mode='w+', dtype=np.float64,
                          shape=(n * (n - 1) // 2,))
            _fill_cosine(D, E)
            stage[0] = 'exact in-place linkage'
            print(f'{method}: clustering in place; no second quadratic array', flush=True)
            Z = _inplace_linkage(D, n, method)
            peak[0] = max(peak[0], process.memory_info().rss)
        finally:
            stop_monitor.set()
            thread.join()
            if D is not None:
                D._mmap.close()
                del D
            Path(filename).unlink(missing_ok=True)
            gc.collect()
        print(f'{method}: distances released; writing outputs', flush=True)
        np.save(out / f'{method}_linkage.npy', Z)
        names.append(f'{method}_linkage.npy')
        for cut in [.4, .5, .6]:
            spec = f'{method}_{cut:.2f}'
            labels = fcluster(Z, cut, criterion='distance')
            assert len(labels) == n
            partitions[spec] = labels
            _, counts = np.unique(labels, return_counts=True)
            rows.append(dict(specification=spec, unique_noise_texts=n,
                             groups=len(counts), singleton_groups=int((counts == 1).sum()),
                             groups_ge5=int((counts >= 5).sum()),
                             texts_in_groups_ge5=int(counts[counts >= 5].sum()),
                             largest_group=int(counts.max())))
            print(rows[-1], flush=True)
        np.savez_compressed(out / 'partitions.npz', **partitions)
        with (out / 'comparison.csv').open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rows[0])
            writer.writeheader()
            writer.writerows(rows)
        execution = dict(method=method, elapsed_seconds=time.time()-started,
                         input_sha256=input_sha256, versions=versions,
                         observed_process_peak_GiB=peak[0]/GIB,
                         conservative_process_peak_GiB=predicted_rss/GIB,
                         conservative_system_use_GiB=predicted_system/GIB,
                         backend='fastcluster 1.3.0 compiled in-place bridge')
        (out / f'{method}_execution.json').write_text(json.dumps(execution, indent=2))
        names.append(f'{method}_execution.json')
        del Z
        gc.collect()
    manifest = dict(status='complete', unique_noise_texts=n, input_sha256=input_sha256,
                    versions=versions, metric='cosine', methods=['average', 'complete'],
                    cuts=[.4, .5, .6], sampling=False, SMDI_assignments_changed=False,
                    backend='fastcluster 1.3.0 compiled in-place bridge',
                    distance_storage='disk-backed condensed float64, 128-row chunks')
    (out / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    (out / 'requirements-executed.txt').write_text(''.join(
        f'{k}=={versions[k]}\n' for k in ['numpy', 'scipy', 'fastcluster', 'psutil']))
    names += ['partitions.npz', 'comparison.csv', 'manifest.json', 'requirements-executed.txt']
    hashes = {name: _sha256(out / name) for name in names}
    dest = out.parent / 'full_noise_colab_results.zip'
    print('Writing checksummed results ZIP...', flush=True)
    with zipfile.ZipFile(dest, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in names:
            z.write(out / name, name)
        z.writestr('SHA256SUMS.json', json.dumps(hashes, indent=2))
    return dest


In [ ]:
result_zip=run_full_ahc(embeddings, '/content/full_noise_results', INPUT_SHA256)
print('Completed both methods on all 35,724 texts:',result_zip)
files.download(str(result_zip))


## After completion
The ZIP contains both linkage trees, all six partitions, comparison counts, input hash and executed package versions. Keep it with this notebook. Bring the ZIP back to the local project: the supplied importer maps all unique texts and 36,592 activity records without uploading their text to Colab. Original BERTopic labels remain unchanged.

If the runtime disconnects, completed method files may be downloaded from the Files sidebar while the VM still exists. This notebook does not mount Drive or promise persistence across VM deletion. A fresh full run recomputes both methods.
